### Load the raw CSV files

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

CUSTOMERS_PATH = "/Volumes/workspace/commerce_dataset/commerce/customers_raw.csv"
ORDERS_PATH = "/Volumes/workspace/commerce_dataset/commerce/orders_raw.csv"
ORDER_ITEMS_PATH = "/Volumes/workspace/commerce_dataset/commerce/order_items_raw.csv"
PRODUCTS_PATH = "/Volumes/workspace/commerce_dataset/commerce/products_raw.csv"
PAYMENTS_PATH = "/Volumes/workspace/commerce_dataset/commerce/payments_raw.csv"

CUSTOMERS_SCHEMA = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("city", T.StringType(), True),
    T.StructField("customer_type", T.StringType(), True)
])

ORDERS_SCHEMA = T.StructType([
    T.StructField("order_id", T.IntegerType(), True),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("order_date", T.StringType(), True),
    T.StructField("shipping_city", T.StringType(), True),
    T.StructField("order_status", T.StringType(), True)
])

ORDER_ITEMS_SCHEMA = T.StructType([
    T.StructField("order_item_id", T.IntegerType(), True),
    T.StructField("order_id", T.IntegerType(), True),
    T.StructField("product_id", T.IntegerType(), True),
    T.StructField("quantity", T.IntegerType(), True),
    T.StructField("unit_price", T.DecimalType(10, 2), True)
])

PRODUCTS_SCHEMA = T.StructType([
    T.StructField("product_id", T.IntegerType(), True),
    T.StructField("product_name", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("list_price", T.DecimalType(10, 2), True)
])

PAYMENTS_SCHEMA = T.StructType([
    T.StructField("payment_id", T.IntegerType(), True),
    T.StructField("order_id", T.IntegerType(), True),
    T.StructField("amount", T.DecimalType(10, 2), True),
    T.StructField("payment_method", T.StringType(), True),
    T.StructField("payment_status", T.StringType(), True),
    T.StructField("payment_date", T.StringType(), True)
])

customers_raw = spark.read.csv(CUSTOMERS_PATH, header=True, schema=CUSTOMERS_SCHEMA)
orders_raw = spark.read.csv(ORDERS_PATH, header=True, schema=ORDERS_SCHEMA)
order_items_raw = spark.read.csv(ORDER_ITEMS_PATH, header=True, schema=ORDER_ITEMS_SCHEMA)
products_raw = spark.read.csv(PRODUCTS_PATH, header=True, schema=PRODUCTS_SCHEMA)
payments_raw = spark.read.csv(PAYMENTS_PATH, header=True, schema=PAYMENTS_SCHEMA)

### Check row counts and schemas

In [0]:
customer_row_count = customers_raw.count()
order_row_count = orders_raw.count()
order_item_row_count = order_items_raw.count()
product_row_count = products_raw.count()
payment_row_count = payments_raw.count()

print("CUSTOMERS DATASET")
print(f"Total records: {customer_row_count:,}")
print(f"Columns: {customers_raw.columns}")
customers_raw.printSchema()

print("\nORDERS DATASET")
print(f"Total records: {order_row_count:,}")
print(f"Columns: {orders_raw.columns}")
orders_raw.printSchema()

print("\nORDER ITEMS DATASET")
print(f"Total records: {order_item_row_count:,}")
print(f"Columns: {order_items_raw.columns}")
order_items_raw.printSchema()

print("\nPRODUCTS DATASET")
print(f"Total records: {product_row_count:,}")
print(f"Columns: {products_raw.columns}")
products_raw.printSchema()

print("\nPAYMENTS DATASET")
print(f"Total records: {payment_row_count:,}")
print(f"Columns: {payments_raw.columns}")
payments_raw.printSchema()


### Preview the datasets

In [0]:
print("Sample customers:")
display(customers_raw.limit(10))

print("Sample orders:")
display(orders_raw.limit(10))

print("Sample order items:")
display(order_items_raw.limit(10))

print("Sample products:")
display(products_raw.limit(10))

print("Sample payments:")
display(payments_raw.limit(10))


### Count null values

In [0]:
customers_null_counts = customers_raw.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in customers_raw.columns])
orders_null_counts = orders_raw.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in orders_raw.columns])
order_items_null_counts = order_items_raw.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in order_items_raw.columns])
products_null_counts = products_raw.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in products_raw.columns])
payments_null_counts = payments_raw.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in payments_raw.columns])

print("NULL COUNTS - CUSTOMERS")
display(customers_null_counts)

print("NULL COUNTS - ORDERS")
display(orders_null_counts)

print("NULL COUNTS - ORDER ITEMS")
display(order_items_null_counts)

print("NULL COUNTS - PRODUCTS")
display(products_null_counts)

print("NULL COUNTS - PAYMENTS")
display(payments_null_counts)


### Check duplicate records and IDs

In [0]:
exact_customer_duplicates = customers_raw.groupBy(customers_raw.columns).count().filter(F.col("count") > 1)
exact_order_duplicates = orders_raw.groupBy(orders_raw.columns).count().filter(F.col("count") > 1)
exact_order_item_duplicates = order_items_raw.groupBy(order_items_raw.columns).count().filter(F.col("count") > 1)
exact_product_duplicates = products_raw.groupBy(products_raw.columns).count().filter(F.col("count") > 1)
exact_payment_duplicates = payments_raw.groupBy(payments_raw.columns).count().filter(F.col("count") > 1)

duplicate_customer_ids = customers_raw.groupBy("customer_id").count().filter(F.col("count") > 1)
duplicate_order_ids = orders_raw.groupBy("order_id").count().filter(F.col("count") > 1)
duplicate_order_item_ids = order_items_raw.groupBy("order_item_id").count().filter(F.col("count") > 1)
duplicate_product_ids = products_raw.groupBy("product_id").count().filter(F.col("count") > 1)
duplicate_payment_ids = payments_raw.groupBy("payment_id").count().filter(F.col("count") > 1)

print(f"Exact duplicated customer records: {exact_customer_duplicates.count():,}")
print(f"Customer IDs appearing more than once: {duplicate_customer_ids.count():,}")
print(f"Exact duplicated order records: {exact_order_duplicates.count():,}")
print(f"Order IDs appearing more than once: {duplicate_order_ids.count():,}")
print(f"Exact duplicated order-item records: {exact_order_item_duplicates.count():,}")
print(f"Order-item IDs appearing more than once: {duplicate_order_item_ids.count():,}")
print(f"Exact duplicated product records: {exact_product_duplicates.count():,}")
print(f"Product IDs appearing more than once: {duplicate_product_ids.count():,}")
print(f"Exact duplicated payment records: {exact_payment_duplicates.count():,}")
print(f"Payment IDs appearing more than once: {duplicate_payment_ids.count():,}")


### Check business values

In [0]:
print("CUSTOMER TYPE VALUES:")
display(customers_raw.groupBy("customer_type").count().orderBy(F.desc("count")))

print("ORDER STATUS VALUES:")
display(orders_raw.groupBy("order_status").count().orderBy(F.desc("count")))

print("PRODUCT CATEGORY VALUES:")
display(products_raw.groupBy("category").count().orderBy(F.desc("count")))

print("PAYMENT METHOD VALUES:")
display(payments_raw.groupBy("payment_method").count().orderBy(F.desc("count")))

print("PAYMENT STATUS VALUES:")
display(payments_raw.groupBy("payment_status").count().orderBy(F.desc("count")))


### Check customer quality

In [0]:
missing_customer_names = customers_raw.filter(F.col("customer_name").isNull() | (F.trim(F.col("customer_name")) == ""))
malformed_customer_emails = customers_raw.filter(F.col("email").isNotNull() & ~F.trim(F.col("email")).rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"))

print(f"Missing customer names: {missing_customer_names.count():,}")
print(f"Malformed customer emails: {malformed_customer_emails.count():,}")

if missing_customer_names.count() > 0:
    display(missing_customer_names.limit(20))

if malformed_customer_emails.count() > 0:
    display(malformed_customer_emails.select("customer_id", "customer_name", "email").limit(20))


### Check order quality

In [0]:
orders_profile = (
    orders_raw
    .withColumn("_order_date_parsed", F.expr("try_cast(order_date as date)"))
    .withColumn("_order_status_normalized", F.lower(F.trim(F.col("order_status"))))
    .withColumn("_order_status_normalized", F.when(F.col("_order_status_normalized") == "canceled", "cancelled").otherwise(F.col("_order_status_normalized")))
)

invalid_order_dates = orders_profile.filter(F.col("_order_date_parsed").isNull())
unexpected_order_statuses = orders_profile.filter(F.col("_order_status_normalized").isNull() | ~F.col("_order_status_normalized").isin("completed", "pending", "cancelled", "refunded"))

print(f"Invalid/missing order dates: {invalid_order_dates.count():,}")
print(f"Unexpected order statuses: {unexpected_order_statuses.count():,}")

if invalid_order_dates.count() > 0:
    display(invalid_order_dates.select("order_id", "order_date", "order_status").limit(20))

if unexpected_order_statuses.count() > 0:
    display(unexpected_order_statuses.select("order_id", "order_status").limit(20))


### Check order-item and product quality

In [0]:
invalid_quantities = order_items_raw.filter(F.col("quantity").isNull() | (F.col("quantity") <= 0))
invalid_unit_prices = order_items_raw.filter(F.col("unit_price").isNull() | (F.col("unit_price") <= 0))
invalid_list_prices = products_raw.filter(F.col("list_price").isNull() | (F.col("list_price") <= 0))

print(f"Invalid/missing order-item quantities: {invalid_quantities.count():,}")
print(f"Invalid/missing order-item unit prices: {invalid_unit_prices.count():,}")
print(f"Invalid/missing product list prices: {invalid_list_prices.count():,}")

if invalid_quantities.count() > 0:
    display(invalid_quantities.limit(20))

if invalid_unit_prices.count() > 0:
    display(invalid_unit_prices.limit(20))


### Check payment quality

In [0]:
payments_profile = (
    payments_raw
    .withColumn("_payment_date_parsed", F.expr("try_cast(payment_date as date)"))
    .withColumn("_payment_status_normalized", F.lower(F.trim(F.col("payment_status"))))
    .withColumn("_payment_method_normalized", F.lower(F.trim(F.col("payment_method"))))
    .withColumn("_payment_method_normalized", F.when(F.col("_payment_method_normalized") == "bank_transfer", "bank transfer").otherwise(F.col("_payment_method_normalized")))
)

invalid_payment_dates = payments_profile.filter(F.col("_payment_date_parsed").isNull())
unexpected_payment_statuses = payments_profile.filter(F.col("_payment_status_normalized").isNull() | ~F.col("_payment_status_normalized").isin("paid", "failed", "refunded"))
unexpected_payment_methods = payments_profile.filter(F.col("_payment_method_normalized").isNull() | ~F.col("_payment_method_normalized").isin("card", "bank transfer", "paypal", "cash"))
invalid_payment_amounts = payments_profile.filter(
    F.col("amount").isNull() |
    ((F.col("_payment_status_normalized") == "paid") & (F.col("amount") <= 0)) |
    ((F.col("_payment_status_normalized") == "failed") & (F.col("amount") != 0)) |
    ((F.col("_payment_status_normalized") == "refunded") & (F.col("amount") >= 0))
)

print(f"Invalid/missing payment dates: {invalid_payment_dates.count():,}")
print(f"Unexpected payment statuses: {unexpected_payment_statuses.count():,}")
print(f"Unexpected payment methods: {unexpected_payment_methods.count():,}")
print(f"Payment amounts inconsistent with status: {invalid_payment_amounts.count():,}")

if invalid_payment_amounts.count() > 0:
    display(invalid_payment_amounts.limit(20))


### Check source relationships

In [0]:
customer_ids = customers_raw.select("customer_id").distinct()
order_ids = orders_raw.select("order_id").distinct()
product_ids = products_raw.select("product_id").distinct()

orders_without_customer = orders_raw.join(customer_ids, on="customer_id", how="left_anti")
items_without_order = order_items_raw.join(order_ids, on="order_id", how="left_anti")
items_without_product = order_items_raw.join(product_ids, on="product_id", how="left_anti")
payments_without_order = payments_raw.join(order_ids, on="order_id", how="left_anti")

print(f"Orders referencing missing customers: {orders_without_customer.count():,}")
print(f"Order items referencing missing orders: {items_without_order.count():,}")
print(f"Order items referencing missing products: {items_without_product.count():,}")
print(f"Payments referencing missing orders: {payments_without_order.count():,}")


### Check payment dates against order dates

In [0]:
orders_for_payment_check = orders_raw.select("order_id", F.expr("try_cast(order_date as date)").alias("_order_date_parsed")).dropDuplicates(["order_id"])

payments_before_order = (
    payments_raw
    .withColumn("_payment_date_parsed", F.expr("try_cast(payment_date as date)"))
    .join(orders_for_payment_check, on="order_id", how="inner")
    .filter(F.col("_payment_date_parsed").isNotNull() & F.col("_order_date_parsed").isNotNull() & (F.col("_payment_date_parsed") < F.col("_order_date_parsed")))
)

print(f"Payments dated before their order date: {payments_before_order.count():,}")

if payments_before_order.count() > 0:
    display(payments_before_order.select("payment_id", "order_id", "amount", "payment_status", "payment_date", "_order_date_parsed").limit(20))


### Create the Bronze DataFrames

In [0]:
customers_bronze = customers_raw.withColumn("_source_file", F.lit(CUSTOMERS_PATH))
orders_bronze = orders_raw.withColumn("_source_file", F.lit(ORDERS_PATH))
order_items_bronze = order_items_raw.withColumn("_source_file", F.lit(ORDER_ITEMS_PATH))
products_bronze = products_raw.withColumn("_source_file", F.lit(PRODUCTS_PATH))
payments_bronze = payments_raw.withColumn("_source_file", F.lit(PAYMENTS_PATH))

print("Bronze DataFrames created.")
print("order_date and payment_date remain strings in Bronze.")


### Save the Bronze tables

In [0]:
customers_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.bronze_customers")
orders_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.bronze_orders")
order_items_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.bronze_order_items")
products_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.bronze_products")
payments_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.bronze_payments")

print("Bronze Delta tables saved successfully.")


### Show the Bronze summary

In [0]:
print("BRONZE SUMMARY")
print(f"Customers: {customers_bronze.count():,}")
print(f"Orders: {orders_bronze.count():,}")
print(f"Order items: {order_items_bronze.count():,}")
print(f"Products: {products_bronze.count():,}")
print(f"Payments: {payments_bronze.count():,}")
